# v0.3 S1-HP — HotpotQA retrieval (E1): sub-corpus, 7 retrievers
**How to run:** Settings → Accelerator **GPU T4 x2**, Internet **On**. Then **Save Version → Save & Run All (Commit)**. It runs unattended (about 3–4 h). If a step fails, the notebook stops with `FAILED` — send me the log. When it finishes, open the version's **Output** tab, download `v03_s1hp_results.zip` and send it to me.

In [ ]:
# Cell 0 — helper: run a shell command, stream its output, STOP the notebook if it fails
import os, subprocess
def sh(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")

In [ ]:
# Cell 0b — run two commands at once (one per GPU); STOP if either fails
import threading
def sh_par(*cmds):
    errs = []
    def run(c):
        try:
            sh(c)
        except Exception as e:
            errs.append(str(e))
    ts = [threading.Thread(target=run, args=(c,)) for c in cmds]
    [t.start() for t in ts]; [t.join() for t in ts]
    if errs:
        raise RuntimeError("; ".join(errs))

In [ ]:
# Cell 0c — the code of commit 1dccbf3 as base64 tar.gz (do not edit; Cell 1 checks it)
CODE = "..."

In [ ]:
# Cell 1 — hardware, code frozen at 1dccbf3 (embedded below), pinned environment, check, tests
sh("nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv || true")
sh("df -h / /tmp /kaggle/working && free -g && nproc")
os.chdir("/kaggle/working")
try:                                   # Hugging Face token from Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN)
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("HF_TOKEN loaded")
except Exception as e:
    print("HF_TOKEN not available (public downloads still work):", type(e).__name__)
# the GitHub repo is private: the code of commit 1dccbf3e959fd4a8408dc9160e5292d2c8761fc6 is embedded in Cell 0c (no dataset, no clone);
# STOP unless the extracted files reproduce that commit's git tree exactly (paper_inputs, paper_outputs left out)
import base64, io, shutil, tarfile
shutil.rmtree("rag", ignore_errors=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE))).extractall("rag")
os.chdir("/kaggle/working/rag")
sh("git init -q && git add -A -f")
sh("test \"$(git write-tree)\" = 585c807c50414d229fa05e76619e7657ca13f1e7 || (echo WRONG CODE; exit 1)")
sh("git -c user.name=kaggle -c user.email=kaggle@local commit -qm 'code 1dccbf3e959fd4a8408dc9160e5292d2c8761fc6' && echo CODE OK 1dccbf3")
sh("pip install -q -r requirements-v03-retrieval.txt")

sh("python -m src.check_env --lock requirements-retrieval.lock --report-only results/v03_hp/raw/check_env_v03_S1hp.json")
sh("python -m src.env_report --session v03_S1hp --out results/v03_hp/raw/env_v03_S1hp.json")
sh("python -m pytest -q -p no:cacheprovider 2>&1 | tail -3")

In [ ]:
# Cell 2 — E1 data: HotpotQA (BEIR corpus 5.2M passages, test qrels, answers), pinned at first download; join by id (gate >= 95%)
sh("python -m src.data.download --config configs/v03_hp_qwen.yaml")
sh("python -m src.data.join_hotpot --config configs/v03_hp_qwen.yaml")

In [ ]:
# Cell 3 — sub-corpus: BM25 over the FULL corpus for all judged queries, then 300,000 passages (gold + BM25 negatives + random)
sh("python -m src.retrieval.bm25 --config configs/v03_hp_qwen.yaml --corpus full --queries judged")
sh("python -m src.data.subcorpus --config configs/v03_hp_qwen.yaml --size 300000 --population judged --name main")

In [ ]:
# Cell 4 — R0: BM25 on the sub-corpus, all judged queries
sh("python -m src.retrieval.bm25 --config configs/v03_hp_qwen.yaml --corpus main --queries judged")

In [ ]:
# Cell 5 — encode the 300k sub-corpus + all judged queries: GPU 0 = R1,R2,R3,R5, GPU 1 = R4,R6 (in parallel, ~1.7 h)
sh_par("CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.encode --config configs/v03_hp_qwen.yaml --models R1,R2,R3,R5 --corpus main",
       "CUDA_VISIBLE_DEVICES=1 python -m src.retrieval.encode --config configs/v03_hp_qwen.yaml --models R4,R6 --corpus main")

In [ ]:
# Cell 6 — exact search (CPU), all judged queries
sh("python -m src.retrieval.search --config configs/v03_hp_qwen.yaml --model R1 --corpus main --queries judged")
sh("python -m src.retrieval.search --config configs/v03_hp_qwen.yaml --model R2 --corpus main --queries judged")
sh("python -m src.retrieval.search --config configs/v03_hp_qwen.yaml --model R3 --corpus main --queries judged")
sh("python -m src.retrieval.search --config configs/v03_hp_qwen.yaml --model R5 --corpus main --queries judged")
sh("python -m src.retrieval.search --config configs/v03_hp_qwen.yaml --model R4 --corpus main --queries judged")
sh("python -m src.retrieval.search --config configs/v03_hp_qwen.yaml --model R6 --corpus main --queries judged")

In [ ]:
# Last cell — package (becomes this version's saved output)
sh("ls results/v03_hp/raw/runs | head -40")
sh("zip -qr /kaggle/working/v03_s1hp_results.zip results/v03_hp data/processed/hp_* configs/resolved_revisions.json logs requirements-*.lock data/raw/hp_qrels/test.tsv data/raw/hp_corpus/queries")
sh("ls -la /kaggle/working/v03_s1hp_results.zip")
sh("rm -rf /kaggle/working/rag/data/raw /kaggle/working/rag/data/embeddings_hp /tmp/models")